# DFU Reliable Run Recovery Search
Non-destructively searches the currently mounted Google Drive for saved reliable-framework checkpoints, completed trials, predictions, and alternate run folders. It does not train, move, overwrite, or delete files.

In [ ]:
import json, os, time
from pathlib import Path
from google.colab import drive

MOUNT=Path('/content/drive')
MY=MOUNT/'MyDrive'
if not MY.is_dir():
    drive.mount(str(MOUNT), force_remount=False)
if not MY.is_dir():
    raise RuntimeError('Google Drive is not mounted.')

targets=[]
interesting={'COMPLETE.json','last_resume.pt','best_model.pt','best_model_portable_fp16.pt','test_predictions.csv','FINAL_VERIFICATION.json','fold_seed_metrics.csv','all_oof_predictions.csv'}
started=time.time()
for root, dirs, files in os.walk(MY):
    root_path=Path(root)
    if root_path.name in {'.Trash','Trash'}:
        dirs[:] = []
        continue
    if root_path.name=='RELIABLE_DFU_CV_V1':
        targets.append({'type':'run_root','path':str(root_path)})
    for name in files:
        if name not in interesting:
            continue
        p=root_path/name
        text=str(p).lower()
        if 'reliable_dfu' not in text and 'reliable-dfu' not in text and 'reliable_dfu_cv_v1' not in text:
            continue
        try:
            size=p.stat().st_size
        except Exception:
            size=None
        targets.append({'type':'artifact','name':name,'path':str(p),'bytes':size})

run_roots=sorted({x['path'] for x in targets if x['type']=='run_root'})
summaries=[]
for root_str in run_roots:
    root=Path(root_str)
    complete=list(root.rglob('COMPLETE.json'))
    resumable=list(root.rglob('last_resume.pt'))
    predictions=list(root.rglob('test_predictions.csv'))
    total_bytes=0
    file_count=0
    for p in root.rglob('*'):
        if p.is_file():
            file_count+=1
            try: total_bytes+=p.stat().st_size
            except Exception: pass
    summaries.append({
        'run_root':str(root),
        'complete_json_count':len(complete),
        'last_resume_count':len(resumable),
        'prediction_csv_count':len(predictions),
        'file_count':file_count,
        'total_bytes':total_bytes,
        'has_final_verification':(root/'FINAL_VERIFICATION.json').exists(),
        'sample_complete':[str(p) for p in complete[:5]],
        'sample_resume':[str(p) for p in resumable[:5]],
    })

report={
    'mounted_root':str(MY),
    'search_seconds':round(time.time()-started,2),
    'run_roots_found':len(run_roots),
    'summaries':summaries,
    'artifact_hits':len([x for x in targets if x['type']=='artifact']),
}
out=MY/'DFU-ImageGuard'/'recovery_audits'/'RELIABLE_DFU_RECOVERY_SEARCH.json'
out.parent.mkdir(parents=True,exist_ok=True)
out.write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps(report,indent=2))
print('Saved:',out)
if not summaries:
    print('RECOVERY RESULT: No reliable-run folder was found in the currently mounted Drive account.')
elif max(x['complete_json_count'] for x in summaries)==0 and max(x['last_resume_count'] for x in summaries)==0:
    print('RECOVERY RESULT: Run folders exist, but no completed or resumable trial artifacts were found.')
else:
    print('RECOVERY RESULT: Usable artifacts were found. Do not start a fresh run until the best source folder is selected.')
